# Stage 10: Business Output
## Meridian Trust Bank — Personal Loan Application Credit Scorecard

**Business question:** we have a validated model — now how does this
actually get USED by the business? This stage builds the final
application-level output table (application ID → PD → score → risk band →
decision) that would feed an underwriting system, and prepares the
analyst-facing Excel workbook and Power BI dashboard data model that let
Risk, underwriting leadership, and portfolio managers actually consume this
work.

This is also where a real analyst's job often ACTUALLY starts to feel
"real" — the modeling is done, and now it needs to be explained, defended,
and operationalized for people who were not in the room for Stages 1-9.

In [1]:
import pandas as pd
import numpy as np
import json

train = pd.read_csv('../01_data/raw_csv/train_scored_final.csv')
test = pd.read_csv('../01_data/raw_csv/test_scored_final.csv')

train['dataset_split'] = 'Train'
test['dataset_split'] = 'Test'

final_output = pd.concat([train, test], ignore_index=True)
print(final_output.shape)
final_output[['application_id', 'predicted_pd', 'credit_score', 'risk_band', 'decision', 'dataset_split']].head()

(5578, 15)


,application_id,predicted_pd,credit_score,risk_band,decision,dataset_split
0,APP008094,0.016730,604.665194,Premium,Approve - Premium Pricing,Train
1,APP007386,0.030292,587.133333,Standard,Approve - Standard Terms,Train
2,APP002766,0.032593,584.953042,Standard,Approve - Standard Terms,Train
3,APP001828,0.042104,577.279920,High Risk - Refer,Refer to Manual Underwriting,Train
4,APP004458,0.100623,550.322087,Decline,Decline,Train


## 10.1 — Enrich the final output with application context

The pure model output (PD, score, band, decision) is only useful alongside
business context — loan amount, purpose, channel — so an underwriter or
portfolio manager can actually interpret and act on it, and so Power BI can
build meaningful cuts (bad rate by channel, by loan purpose, etc.).

In [2]:
modeling = pd.read_csv('../01_data/raw_csv/modeling_dataset.csv')

context_cols = ['application_id', 'application_date', 'loan_purpose', 'loan_amount_requested',
                 'term_months', 'channel', 'state', 'employment_type', 'annual_income']

final_output = final_output.merge(modeling[context_cols], on='application_id', how='left')

# Round PD and score for presentation
final_output['predicted_pd'] = final_output['predicted_pd'].round(4)
final_output['credit_score'] = final_output['credit_score'].round(0).astype(int)

final_output = final_output[[
    'application_id', 'application_date', 'dataset_split',
    'loan_purpose', 'loan_amount_requested', 'term_months', 'channel', 'state',
    'employment_type', 'annual_income',
    'predicted_pd', 'credit_score', 'risk_band', 'decision', 'bad_flag'
]]

final_output.head(10)

,application_id,application_date,dataset_split,loan_purpose,loan_amount_requested,term_months,channel,state,employment_type,annual_income,predicted_pd,credit_score,risk_band,decision,bad_flag
0,APP008094,2023-07-02,Train,Debt Consolidation,22500,48,Online,QLD,Self-Employed,44520.19,0.0167,605,Premium,Approve - Premium Pricing,0
1,APP007386,2023-11-11,Train,Wedding,9500,24,Branch,SA,Full-Time,122895.39,0.0303,587,Standard,Approve - Standard Terms,0
2,APP002766,2023-09-11,Train,Debt Consolidation,22000,48,Broker,VIC,Part-Time,47660.28,0.0326,585,Standard,Approve - Standard Terms,0
3,APP001828,2023-08-18,Train,Travel,43500,24,Branch,WA,Self-Employed,15000.00,0.0421,577,High Risk - Refer,Refer to Manual Underwriting,0
4,APP004458,2023-03-03,Train,Home Renovation,13000,24,Online,NSW,Full-Time,58972.90,0.1006,550,Decline,Decline,0
5,APP007061,2023-07-03,Train,Debt Consolidation,28000,24,Broker,NSW,Self-Employed,59712.83,0.0500,572,High Risk - Refer,Refer to Manual Underwriting,0
6,APP006170,2023-05-03,Train,Debt Consolidation,33000,36,Branch,NSW,Full-Time,103484.92,0.0189,601,Preferred,Approve - Preferred Pricing,0
7,APP005922,2023-11-16,Train,Wedding,15500,36,Branch,QLD,Casual,45041.11,0.0296,588,Standard,Approve - Standard Terms,0
8,APP004708,2023-05-20,Train,Car Purchase,29000,36,Online,NSW,Full-Time,70207.73,0.0344,583,High Risk - Refer,Refer to Manual Underwriting,0
9,APP008964,2023-03-16,Train,Home Renovation,24500,12,Online,VIC,Full-Time,45661.73,0.0258,592,Standard,Approve - Standard Terms,0


## 10.2 — Portfolio-level summary (what a Risk report would lead with)

In [3]:
portfolio_summary = final_output.groupby('risk_band', observed=True).agg(
    n_applications=('application_id', 'count'),
    avg_predicted_pd=('predicted_pd', 'mean'),
    observed_bad_rate=('bad_flag', 'mean'),
    avg_score=('credit_score', 'mean'),
    avg_loan_amount=('loan_amount_requested', 'mean'),
).round(4)

band_order = ['Decline', 'High Risk - Refer', 'Standard', 'Preferred', 'Premium']
portfolio_summary = portfolio_summary.loc[band_order]
portfolio_summary

,n_applications,avg_predicted_pd,observed_bad_rate,avg_score,avg_loan_amount
risk_band,,,,,
Decline,560,0.1066,0.0946,551.0143,26249.1071
High Risk - Refer,1338,0.0426,0.0478,577.3490,26334.8281
Standard,2038,0.0269,0.0260,590.9598,25535.0834
Preferred,1385,0.0188,0.0181,601.2173,26739.7112
Premium,257,0.0164,0.0311,605.5759,24219.8444


In [4]:
# NOTE: "approval" should mean an automated APPROVE decision, not simply
# "not Decline" -- Refer applications are neither approved nor declined, they
# go to manual underwriting. The first version of this calculation defined
# approval_rate as (decision != 'Decline'), which incorrectly counted Refer
# applications as approvals. Fixed below to count only the three explicit
# "Approve - ..." decision categories.
auto_approved = final_output['decision'].str.startswith('Approve')
auto_approval_rate = auto_approved.mean()
decline_rate = (final_output['decision'] == 'Decline').mean()
refer_rate = (final_output['decision'] == 'Refer to Manual Underwriting').mean()

overall_bad_rate = final_output['bad_flag'].mean()
print(f"Automated approval rate: {auto_approval_rate:.1%}")
print(f"Automated decline rate:  {decline_rate:.1%}")
print(f"Referred to manual underwriting: {refer_rate:.1%}")
print(f"\nOverall portfolio bad rate: {overall_bad_rate:.4f}")
print(f"\nApplications by decision:")
print(final_output['decision'].value_counts())

Automated approval rate: 66.0%
Automated decline rate:  10.0%
Referred to manual underwriting: 24.0%

Overall portfolio bad rate: 0.0364

Applications by decision:
decision
Approve - Standard Terms        2038
Approve - Preferred Pricing     1385
Refer to Manual Underwriting    1338
Decline                          560
Approve - Premium Pricing        257
Name: count, dtype: int64


## 10.3 — Save the final business output CSV

This is the table that would be handed to underwriting systems, Risk
reporting, and Power BI.

In [5]:
final_output.to_csv('../06_deliverables/final_scored_applications.csv', index=False)
print(f"Saved {len(final_output)} scored applications to 06_deliverables/final_scored_applications.csv")

Saved 5578 scored applications to 06_deliverables/final_scored_applications.csv


## 10.4 — Checkpoint questions

1. `dataset_split` (Train/Test) is included in this final output. Should
   this column be visible to underwriters/business users, or is it purely
   an internal modeling artifact that should be stripped before handing
   this off? What's the argument each way?

2. The portfolio summary in 10.2 shows `avg_loan_amount` by risk band. If
   "Premium" (lowest risk) applicants also tend to request LARGER loans, and
   "Decline" applicants tend to request SMALLER ones, what would that imply
   about MTB's total dollar exposure concentration by risk band — and why
   would a portfolio manager care about this beyond just the applicant
   counts we've been focused on throughout this project?

3. We built ONE static scorecard from ONE point-in-time development sample.
   In production, how would you know when it's time to redevelop this
   scorecard rather than keep using it? (Connect this back to Stage 9's PSI
   discussion, and think about what OTHER signals — beyond score
   distribution drift — might also indicate the model needs refreshing.)